导入库和加载数据

In [28]:
import pandas as pd
import jieba
from gensim.models.word2vec import Word2Vec

data = pd.read_csv(r"D:\VS Code Program\NLP\train.csv")
data.head()

,label,comment
0,0,一如既往地好吃，希望可以开到其他城市
1,0,味道很不错，分量足，客人很多，满意
2,0,下雨天来的，没有想象中那么火爆。环境非常干净，古色古香的，我自己也是个做服务行业的，我都觉得...
3,0,真心不好吃 基本上没得好多味道
4,0,少送一个牛肉汉堡 而且也不好吃 特别是鸡肉卷 **都不想评论了 谁买谁知道


数据预处理和分词

In [29]:
corpus = data['comment'].values.astype(str)
corpus = [jieba.lcut(corpus[index]
                          .replace("，", "")
                          .replace("!", "")
                          .replace("！", "")
                          .replace("。", "")
                          .replace("~", "")
                          .replace("；", "")
                          .replace("？", "")
                          .replace("?", "")
                          .replace("【", "")
                          .replace("】", "")
                          .replace("#", "")
                        ) for index in range(len(corpus))]

corpus[0]

['一如既往', '地', '好吃', '希望', '可以', '开', '到', '其他', '城市']

训练Word2Vec模型

In [30]:
model = Word2Vec(corpus, sg=0, vector_size=300, window=5, min_count=3, workers=4)
model

最匹配词

In [31]:
most_similar = model.wv.most_similar(positive=['好吃'], negative=['难吃'])
most_similar_words = [word for word, _ in most_similar]
print('最匹配的词是：', most_similar_words)

最匹配的词是： ['现做现', '那儿', '氛围', '位置', '主食', '热情', '团购', '好量', '非常', '好味']


语义相似度（美味-好吃）

In [32]:
similarity1 = model.wv.similarity('美味', '好吃')
print('美味, 好吃的相似度为=', similarity1)

美味, 好吃的相似度为= 0.8412361


语义相似度（好吃-蟑螂）

In [33]:
similarity2 = model.wv.similarity('好吃', '蟑螂')
print('好吃, 蟑螂的相似度为=', similarity2)

好吃, 蟑螂的相似度为= 0.3930572


获取词向量

In [34]:
word_vector = model.wv.__getitem__('地道')
print("词语'地道'的词向量（前10个维度）:")
print(word_vector[:10])
print(f"向量维度: {len(word_vector)}")

词语'地道'的词向量（前10个维度）:
[ 0.00172181  0.08419146  0.02538798  0.08853555 -0.04094086 -0.12187498
  0.06546208  0.2985202   0.0205784  -0.0433825 ]
向量维度: 300


向量运算

In [40]:
result = model.wv.most_similar(positive=['餐厅', '聚会'], negative=['安静'], topn=5)
print("餐厅 + 聚会 - 安静 = ?")
for word, similarity in result:
    print(f"{word}: {similarity:.4f}")

餐厅 + 聚会 - 安静 = ?
生日: 0.9937
想起: 0.9932
最好: 0.9931
定: 0.9930
一楼: 0.9923
